In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Transform to convert images to tensors (scales pixels to [0, 1])
transform = transforms.ToTensor()

# Load MNIST Training Data
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=128, shuffle=True)

Using device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 62.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.72MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.1MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.31MB/s]


In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):
        super(VAE, self).__init__()

        # 1. ENCODER
        # First layer to compress the input
        self.fc1 = nn.Linear(input_dim, hidden_dim)

        # Instead of a single output, the encoder outputs TWO vectors for the latent space:
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)      # Calculates Mean (μ)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)  # Calculates Log Variance (log(σ^2))

        # 3. DECODER
        # Expands the latent vector 'z' back to the hidden dimension
        self.fc3 = nn.Linear(latent_dim, hidden_dim)
        # Expands back to the original image size (784)
        self.fc4 = nn.Linear(hidden_dim, input_dim)

    # 2. REPARAMETERIZATION TRICK
    def reparameterize(self, mu, logvar):
        
        # Convert log variance to standard deviation: σ = exp(0.5 * log(σ^2))
        std = torch.exp(0.5 * logvar)
        # Sample random noise (epsilon) of the same shape as std
        eps = torch.randn_like(std)
        # Calculate and return the latent vector z
        return mu + eps * std

    def forward(self, x):
        # Encode
        h1 = F.relu(self.fc1(x))
        mu = self.fc_mu(h1)
        logvar = self.fc_logvar(h1)

        # Reparameterize
        z = self.reparameterize(mu, logvar)

        # Decode
        h3 = F.relu(self.fc3(z))

        reconstructed_x = torch.sigmoid(self.fc4(h3))

        # Returns the reconstructed image, plus the mean and logvar (needed for the loss function)
        return reconstructed_x, mu, logvar

# Instantiate the model
model = VAE().to(device)

In [3]:
# Define the Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Define the custom VAE Loss Function
def vae_loss_function(recon_x, x, mu, logvar):
    # 1. Reconstruction Loss (Binary Cross Entropy), reduction='sum' adds up the loss for all 784 pixels
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')

    # 2. KL Divergence Loss
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    # total loss
    return BCE + KLD, BCE, KLD

# Training Loop
epochs = 10

for epoch in range(epochs):
    model.train()
    train_loss = 0
    total_bce = 0
    total_kld = 0

    for batch_idx, (data, _) in enumerate(train_loader):
        # Flatten the 28x28 MNIST images into 784-dimensional vectors
        data = data.view(-1, 784).to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass through the VAE
        recon_batch, mu, logvar = model(data)

        # Calculate Reconstruction loss, KL divergence loss, and Total loss
        loss, bce, kld = vae_loss_function(recon_batch, data, mu, logvar)

        # Backward pass to calculate gradients
        loss.backward()

        # Update model weights
        optimizer.step()

        # Accumulate losses for reporting
        train_loss += loss.item()
        total_bce += bce.item()
        total_kld += kld.item()

    # Output the loss values for the epoch
    avg_loss = train_loss / len(train_loader.dataset)
    avg_bce = total_bce / len(train_loader.dataset)
    avg_kld = total_kld / len(train_loader.dataset)

    print(f"Epoch [{epoch+1}/{epochs}] | "
          f"Total Loss: {avg_loss:.2f} | "
          f"Recon Loss (BCE): {avg_bce:.2f} | "
          f"KL Div Loss: {avg_kld:.2f}")

print("\nVAE Training Complete!")

Epoch [1/10] | Total Loss: 164.84 | Recon Loss (BCE): 149.03 | KL Div Loss: 15.80
Epoch [2/10] | Total Loss: 121.41 | Recon Loss (BCE): 98.64 | KL Div Loss: 22.77
Epoch [3/10] | Total Loss: 114.69 | Recon Loss (BCE): 90.37 | KL Div Loss: 24.32
Epoch [4/10] | Total Loss: 111.77 | Recon Loss (BCE): 87.00 | KL Div Loss: 24.77
Epoch [5/10] | Total Loss: 110.06 | Recon Loss (BCE): 85.07 | KL Div Loss: 24.99
Epoch [6/10] | Total Loss: 108.92 | Recon Loss (BCE): 83.81 | KL Div Loss: 25.11
Epoch [7/10] | Total Loss: 108.04 | Recon Loss (BCE): 82.87 | KL Div Loss: 25.16
Epoch [8/10] | Total Loss: 107.47 | Recon Loss (BCE): 82.20 | KL Div Loss: 25.27
Epoch [9/10] | Total Loss: 106.90 | Recon Loss (BCE): 81.63 | KL Div Loss: 25.27
Epoch [10/10] | Total Loss: 106.47 | Recon Loss (BCE): 81.14 | KL Div Loss: 25.33

VAE Training Complete!


The implemented Variational Autoencoder (VAE) successfully transforms high-dimensional MNIST digits into a compressed, continuous probabilistic latent space defined by a learned mean ($\mu$) and variance ($\sigma^2$). By utilizing the reparameterization trick, the model can randomly sample from this distribution while keeping the entire network differentiable for backpropagation. The custom training loop actively balances two competing objectives: the Reconstruction Loss ensures the decoder accurately rebuilds the original image, while the KL Divergence Loss acts as a mathematical regulator, forcing the latent space to resemble a standard normal distribution. This balance prevents the model from simply memorizing the training data, resulting in a perfectly organized latent space that allows the VAE to generate entirely new, realistic handwritten digits.